<a href="https://colab.research.google.com/github/YOUR-USERNAME/bags-vectors-transformers/blob/main/day3/notebooks/1_bert_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Bags, Vectors & Transformers
## Day 3 — Contextual Embeddings with BERT  ·  **SOLUTIONS**

**A Methods Workshop in Computational Text Analysis**
Denise J. Roth · Strategic Communication Group · Wageningen University & Research

---

This is the **solutions** version. Every **✏️ Exercise** is filled in with one possible answer
plus a short comment. Model outputs vary slightly by version, so your exact numbers may differ.

> **Turn on the GPU!** *Runtime → Change runtime type → T4 GPU.*


## 0. Setup

In [ ]:
!pip install transformers datasets -q

import torch
from transformers import pipeline, AutoTokenizer, AutoModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device.upper())
if device == "cpu":
    print("⚠️  No GPU detected. Things will be slow.")

## 1. The easiest possible start: `pipeline`

In [ ]:
sentiment = pipeline("sentiment-analysis")
result = sentiment("This new policy is an absolute disaster.")
print(result)

In [ ]:
examples = [
    "I love the new public transport plan!",
    "This is the worst decision the city has ever made.",
    "The meeting is scheduled for Tuesday afternoon.",
    "Well, that went just great.",
]
for text in examples:
    r = sentiment(text)[0]
    print(f"{r['label']:<9} ({r['score']:.2f})  {text}")

> **✏️ Exercise 1**
>
> Write three sentences of your own — ideally ambiguous or sarcastic — and run them through
> `sentiment`. Where does it do well? Where does it stumble?


In [ ]:
# ✅ Solution
my_sentences = [
    "The policy is fine, I guess.",                     # lukewarm / ambiguous
    "Oh fantastic, another tax increase.",              # sarcasm
    "The committee will review the proposal next week.", # neutral / factual
]
for s in my_sentences:
    r = sentiment(s)[0]
    print(f"{r['label']:<9} ({r['score']:.2f})  {s}")

# Comment: the model handles clear sentiment well, but it typically MISREADS the sarcastic
# line ("Oh fantastic...") as POSITIVE — it keys on "fantastic" and misses the ironic tone.
# It also tends to force neutral/factual sentences into POSITIVE or NEGATIVE, because this
# particular model only has two labels. These are real limitations: sarcasm and neutrality
# are hard, and the label set constrains what the model can say.

## 2. Under the hood: tokenization

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
text = "Transformers are powerful."
tokens = tokenizer.tokenize(text)
print("Text:  ", text)
print("Tokens:", tokens)

In [ ]:
for word in ["preprocessing", "unbelievable", "gezondheidszorg", "antidisestablishmentarianism"]:
    print(f"{word:<32} -> {tokenizer.tokenize(word)}")

### Special tokens and IDs

In [ ]:
encoded = tokenizer("Transformers are powerful.")
ids = encoded["input_ids"]
print("Token IDs:  ", ids)
print("Back to tokens:", tokenizer.convert_ids_to_tokens(ids))

### The 512-token limit

In [ ]:
print("Max input length for this model:", tokenizer.model_max_length, "tokens")
long_text = "policy " * 20
enc = tokenizer(long_text, max_length=10, truncation=True)
print("Number of tokens kept:", len(enc["input_ids"]))
print("Tokens:", tokenizer.convert_ids_to_tokens(enc["input_ids"]))

> **✏️ Exercise 2**
>
> Tokenize a Dutch sentence with this English tokenizer and see how it splits. What does that
> tell you about wanting a Dutch model?


In [ ]:
# ✅ Solution
dutch = "De nieuwe wet is een ramp"
print("Dutch sentence:", dutch)
print("Tokens:", tokenizer.tokenize(dutch))

# Comment: the English tokenizer over-splits Dutch words into odd subword fragments
# (e.g. "wet", "ramp" may survive by luck, but many Dutch words shatter into pieces
# that carry no meaning). Because the tokenizer's vocabulary was learned from ENGLISH
# text, it has no good subword units for Dutch. A Dutch model like BERTje ships a
# tokenizer trained on Dutch, so words split into meaningful pieces — which is why you
# should use a language-matched model for non-English text.

## 3. The payoff: the "bank" test

In [ ]:
model = AutoModel.from_pretrained("distilbert-base-uncased").to(device)
model.eval()
print("Model loaded.")

In [ ]:
def get_word_vector(sentence, target_word):
    """Return BERT's contextual vector for target_word in a given sentence."""
    enc = tokenizer(sentence, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model(**enc)
    hidden = output.last_hidden_state[0]
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    target_tokens = tokenizer.tokenize(target_word)
    for i, t in enumerate(tokens):
        if t == target_tokens[0]:
            return hidden[i].cpu().numpy()
    raise ValueError(f"'{target_word}' not found in tokens: {tokens}")

s1 = "I sat on the grassy bank of the river."
s2 = "I deposited my paycheck at the bank."
v1 = get_word_vector(s1, "bank")
v2 = get_word_vector(s2, "bank")
print("Got two vectors for 'bank', each of size", v1.shape[0])

In [ ]:
import numpy as np

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

diff_sense = cosine(v1, v2)
s3 = "We fished from the bank of the stream."
v3 = get_word_vector(s3, "bank")
same_sense = cosine(v1, v3)

print(f"'bank' river  vs  'bank' money  (different sense): {diff_sense:.3f}")
print(f"'bank' river  vs  'bank' stream (same sense):      {same_sense:.3f}")

> **✏️ Exercise 3**
>
> Try the same test with another ambiguous word: "spring" or "left". Does BERT separate
> the senses?


In [ ]:
# ✅ Solution — "spring" in three senses
s_season = "Flowers bloom in the spring."
s_coil   = "The mattress spring finally broke."
s_jump   = "The cat will spring onto the table."

v_season = get_word_vector(s_season, "spring")
v_coil   = get_word_vector(s_coil, "spring")
v_jump   = get_word_vector(s_jump, "spring")

print(f"season vs coil: {cosine(v_season, v_coil):.3f}")
print(f"season vs jump: {cosine(v_season, v_jump):.3f}")
print(f"coil   vs jump: {cosine(v_coil, v_jump):.3f}")

# Comment: the three vectors are clearly LESS than 1.0 apart from each other — BERT gives
# "spring" a different representation in each sense, driven by the surrounding words
# (bloom/flowers vs mattress/broke vs cat/onto). A static embedding would give an identical
# vector all three times (similarity 1.000). This is exactly the contextual advantage.

## 4. Back to policy: zero-shot classification

In [ ]:
zero_shot = pipeline("zero-shot-classification")

bill_title = "A bill to expand access to affordable health insurance coverage."
candidate_labels = ["health", "education", "taxation", "defense", "environment"]

result = zero_shot(bill_title, candidate_labels)
print("Title:", bill_title, "\n")
for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:<12} {score:.3f}")

> **✏️ Exercise 4**
>
> Run zero-shot classification on a few bill titles of your own, including one that could fit
> two areas.


In [ ]:
# ✅ Solution
titles = [
    "A bill to fund renewable energy research.",             # environment
    "An act to reduce corporate tax rates.",                 # taxation
    "A bill to fund environmental education in schools.",    # education + environment!
]
for title in titles:
    res = zero_shot(title, candidate_labels)
    top2 = list(zip(res["labels"], res["scores"]))[:2]
    print(title)
    print(f"   {top2[0][0]} ({top2[0][1]:.2f}),  {top2[1][0]} ({top2[1][1]:.2f})\n")

# Comment: the ambiguous third title splits its probability between "education" and
# "environment" rather than committing hard to one — which is exactly what we'd want.
# Zero-shot classification is remarkably useful when you have categories in mind but no
# labeled data. Its weakness: it depends heavily on how you phrase the candidate labels.

## 5. Exploring parameters: model size and speed

In [ ]:
import time

text = "The government announced a major new economic reform package today."
small = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

start = time.time()
for _ in range(20):
    small(text)
small_time = time.time() - start
print(f"DistilBERT (small): {small_time:.2f}s for 20 runs")

### Tokenization parameters that matter

In [ ]:
sample = "This is a fairly long sentence that we will use to demonstrate truncation and padding."
enc_trunc = tokenizer(sample, max_length=8, truncation=True)
print("Truncated to 8 tokens:", tokenizer.convert_ids_to_tokens(enc_trunc["input_ids"]))
print()
batch = tokenizer(["short text", "a somewhat longer piece of text here"],
                  padding=True, truncation=True)
for ids in batch["input_ids"]:
    print(f"length {len(ids)}:", tokenizer.convert_ids_to_tokens(ids))

> **✏️ Exercise 5**
>
> Tokenize a long title with `max_length` set to 5, 15, and 30. How would truncating too
> aggressively hurt a classifier?


In [ ]:
# ✅ Solution
long_title = ("A comprehensive bill to reform the national health insurance system, "
              "expand rural hospital funding, and lower prescription drug prices.")

for max_len in [5, 15, 30]:
    enc = tokenizer(long_title, max_length=max_len, truncation=True)
    kept = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    print(f"max_length={max_len:>2}: {len(kept)} tokens -> {' '.join(kept)}\n")

# Comment: with max_length=5 the title is cut to just "[CLS] a comprehensive bill to [SEP]"
# — the words that actually reveal the policy area ("health", "insurance", "hospital",
# "drug") are CHOPPED OFF. A classifier fed only the truncated version would be guessing.
# Truncation is necessary for long documents, but truncate too hard and you throw away the
# signal. The fix for genuinely long texts is to split them into chunks rather than discard.

## 6. *(Optional)* Fine-tuning on the policy data

> ⚠️ **Needs a GPU** and takes a few minutes.


In [ ]:
from datasets import load_dataset

bills = load_dataset("dreamproit/bill_labels_us", split="train").to_pandas()
bills = bills.rename(columns={"title": "text"})[["text", "policy_area"]].dropna()

top_areas = bills["policy_area"].value_counts().head(4).index.tolist()
bills = bills[bills["policy_area"].isin(top_areas)]
bills = bills.groupby("policy_area", group_keys=False).apply(
    lambda g: g.sample(min(len(g), 400), random_state=42)
).reset_index(drop=True)

labels = sorted(bills["policy_area"].unique())
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}
bills["label"] = bills["policy_area"].map(label2id)

print("Fine-tuning on", len(bills), "bills across", len(labels), "policy areas:")
print(labels)

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)

train_df, test_df = train_test_split(bills, test_size=0.2, random_state=42,
                                     stratify=bills["label"])

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=32)

train_ds = Dataset.from_pandas(train_df).map(tokenize, batched=True)
test_ds = Dataset.from_pandas(test_df).map(tokenize, batched=True)

clf_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
)
print("Ready to fine-tune.")

In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}

args = TrainingArguments(
    output_dir="./bert-policy",
    num_train_epochs=2,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    eval_strategy="epoch",
    logging_steps=20,
    report_to="none",
)

trainer = Trainer(
    model=clf_model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)
trainer.train()

In [ ]:
metrics = trainer.evaluate()
print(f"Fine-tuned accuracy: {metrics['eval_accuracy']:.3f}")

In [ ]:
fine_tuned = pipeline("text-classification", model=clf_model, tokenizer=tokenizer)
print(fine_tuned("A bill to increase funding for public schools and teachers."))

> **✏️ Exercise 6** *(optional)*
>
> Change `num_train_epochs` to 1 and 4, and `learning_rate` to `5e-5`. How does accuracy change?


In [ ]:
# ✅ Solution — a small sweep over hyperparameters
for epochs, lr in [(1, 2e-5), (4, 2e-5), (2, 5e-5)]:
    args = TrainingArguments(
        output_dir=f"./sweep-{epochs}-{lr}",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        learning_rate=lr,
        eval_strategy="no",
        logging_strategy="no",
        report_to="none",
    )
    m = AutoModelForSequenceClassification.from_pretrained(
        "distilbert-base-uncased", num_labels=len(labels),
        id2label=id2label, label2id=label2id)
    t = Trainer(model=m, args=args, train_dataset=train_ds,
                eval_dataset=test_ds, compute_metrics=compute_metrics)
    t.train()
    acc = t.evaluate()["eval_accuracy"]
    print(f"epochs={epochs}, lr={lr}: accuracy = {acc:.3f}")

# Comment: typically more epochs helps up to a point, then plateaus or slightly overfits;
# a higher learning rate (5e-5) can train faster but is less stable and sometimes worse.
# There is NO universally best setting — it depends on your data and model. This is why you
# always validate on a held-out set and report what you actually tried, rather than assuming
# the defaults are optimal.

## Wrap-up

That's the full solution set. The recurring themes: contextual models capture meaning that
static ones cannot (the "bank" test), off-the-shelf tools go a long way before you need to
train anything, and — as always — you validate on held-out data rather than trusting defaults.

Congratulations: you've gone from **bags** of words to **vectors** to **transformers**, with a
working feel for when each belongs in your research.
